# ERA5 vs Published Ground-Based Data Comparison

This notebook compares the ERA5 reanalysis wind data with high-resolution ground-based meteorological data from the ALMA weather station at Chajnantor Plateau for 2001 and 2002.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load datasets
era5_path = r'../data/exports/chile_wind_stats_2001_2002.csv'
published_path = r'../data/exports/alma_published_stats.csv'

df_era5 = pd.read_csv(era5_path)
df_pub = pd.read_csv(published_path)

# Filter ERA5 to just ALMA region if multiple sites exist
if 'site' in df_era5.columns:
    df_era5_alma = df_era5[df_era5['site'] == 'chile'].copy()
else:
    df_era5_alma = df_era5.copy()

# Ensure columns for merging
df_era5_alma['date'] = pd.to_datetime(df_era5_alma[['year', 'month']].assign(day=1))
df_pub['date'] = pd.to_datetime(df_pub[['year', 'month']].assign(day=1))

comparison_df = pd.merge(df_era5_alma, df_pub, on=['year', 'month', 'date'], suffixes=('_era5', '_pub'))
comparison_df.head()

## Monthly Mean Wind Speed Comparison

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(comparison_df['date'], comparison_df['mean_era5'], marker='o', label='ERA5 (ALMA Region)')
plt.plot(comparison_df['date'], comparison_df['mean_pub'], marker='s', label='Ground-Based (ESO Published)')

plt.title('Monthly Mean Wind Speed (2001-2002): ERA5 vs Ground-Based')
plt.xlabel('Date')
plt.ylabel('Wind Speed (m/s)')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../data/exports/era5_vs_pub_means.png')
plt.show()

## Percentile Comparison (P50, P90, P95, P99)

In [ ]:
percentiles = ['p50', 'p90', 'p95', 'p99']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, p in enumerate(percentiles):
    axes[i].plot(comparison_df['date'], comparison_df[f'{p}_era5'], marker='o', label=f'ERA5 {p.upper()}')
    axes[i].plot(comparison_df['date'], comparison_df[f'{p}_pub'], marker='s', label=f'Published {p.upper()}')
    axes[i].set_title(f'Comparison of {p.upper()}')
    axes[i].set_xlabel('Date')
    axes[i].set_ylabel('Wind Speed (m/s)')
    axes[i].legend()
    axes[i].grid(True)
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../data/exports/era5_vs_pub_percentiles.png')
plt.show()

## Statistical Correlations

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(comparison_df['mean_pub'], comparison_df['mean_era5'], alpha=0.7)

# 1:1 line
max_val = max(comparison_df['mean_era5'].max(), comparison_df['mean_pub'].max())
plt.plot([0, max_val], [0, max_val], 'r--', label='1:1 Line')

plt.title('Correlation: Published Mean vs ERA5 Mean')
plt.xlabel('Ground-Based Mean (m/s)')
plt.ylabel('ERA5 Mean (m/s)')
plt.legend()
plt.grid(True)

corr = comparison_df['mean_era5'].corr(comparison_df['mean_pub'])
plt.text(1, max_val-1, f'Pearson Correlation: {corr:.2f}', fontsize=12)
plt.show()